# Pandas: Modify

## Create new columns derived from existing columns

![](../assets/pandas/create_new_columns.png)

In [1]:
import pandas as pd

Let's demonstrate calculate the BMI given weight and height of each athlete:

In [ ]:
# Best practice: use pathlib to handle file paths
from pathlib import Path

# Set the path to the directory containing the datasets
data_dir = Path('.')

# Debugging
print(data_dir.resolve())
print(data_dir.exists())

/home/halgoz/work/Bootcamp/ai-pros/public/courses/01_Fundamentals/data
True


In [3]:
data_socr = data_dir / 'SOCR-HeightWeight.csv'

# Debugging
print(data_socr.exists())
print(data_socr.resolve())

True
/home/halgoz/work/Bootcamp/ai-pros/public/courses/01_Fundamentals/data/SOCR-HeightWeight.csv


**BMI formula**:

$$
BMI = \frac{\text{Weight}}{\text{Height}^2}
$$

In [15]:
df_socr = pd.read_csv(data_socr, index_col='Index')
df_socr.head()

,Height(Inches),Weight(Pounds)
Index,,
1,65.78331,112.9925
2,71.51521,136.4873
3,69.39874,153.0269
4,68.21660,142.3354
5,67.78781,144.2971


### Vector-scalar operations

Since the Height and Weight are in pounds, and the formula assumes **Metric System**, we need to convert the **Imperial System** numbers first:

In [16]:
df_socr['Height(meters)'] = df_socr['Height(Inches)'] * 0.0254
df_socr['Weight(kilograms)'] = df_socr['Weight(Pounds)'] * 0.453592
df_socr.head()

,Height(Inches),Weight(Pounds),Height(meters),Weight(kilograms)
Index,,,,
1,65.78331,112.9925,1.670896,51.252494
2,71.51521,136.4873,1.816486,61.909547
3,69.39874,153.0269,1.762728,69.411778
4,68.21660,142.3354,1.732702,64.562199
5,67.78781,144.2971,1.721810,65.452010


Notice how we are multiplying a **scalar** like `0.0254` by a **vector** (`Series`). This is done **element-wise** (on each element), known as **Broadcasting** the operation.

### Row-wise calculations

Then actually do the calculation:

In [17]:
df_socr['BMI'] = df_socr['Weight(kilograms)'] / (df_socr['Height(meters)'] ** 2)
df_socr.head()

,Height(Inches),Weight(Pounds),Height(meters),Weight(kilograms),BMI
Index,,,,,
1,65.78331,112.9925,1.670896,51.252494,18.357609
2,71.51521,136.4873,1.816486,61.909547,18.762615
3,69.39874,153.0269,1.762728,69.411778,22.338940
4,68.21660,142.3354,1.732702,64.562199,21.504569
5,67.78781,144.2971,1.721810,65.452010,22.077625


Notice how this involves calculation **across two columns of the corresponding row**.

### `apply` functions to calculate values

**BMI table for adults**. This is the World Health Organization's (WHO) recommended body weight based on BMI values for adults. It is used for both men and women, age 20 or older:

| Classification    | BMI range - kg/m2 |
| ----------------- | ----------------- |
| Severe Thinness   | < 16              |
| Moderate Thinness | 16 - 17           |
| Mild Thinness     | 17 - 18.5         |
| Normal            | 18.5 - 25         |
| Overweight        | 25 - 30           |
| Obese Class I     | 30 - 35           |
| Obese Class II    | 35 - 40           |
| Obese Class III   | > 40              |

Source: https://www.calculator.net/bmi-calculator.html

We can code this mapping as a pure Python function:

In [18]:
def get_bmi_category(bmi: float) -> str:
    if bmi < 16:
        return 'Severe Thinness'
    elif bmi < 17:
        return 'Moderate Thinness'
    elif bmi < 18.5:
        return 'Mild Thinness'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    elif bmi < 35:
        return 'Obese Class I'
    elif bmi < 40:
        return 'Obese Class II'
    else:
        return 'Obese Class III'

Then use `apply` to process each value in the column, as follows:

In [19]:
df_socr['classification'] = df_socr['BMI'].apply(get_bmi_category)

Let's sample (randomly) 10 rows to see their classifications:

In [20]:
df_socr.sample(10)

,Height(Inches),Weight(Pounds),Height(meters),Weight(kilograms),BMI,classification
Index,,,,,,
847,70.01279,121.3609,1.778325,55.048333,17.406922,Mild Thinness
24766,69.70400,132.2823,1.770482,60.002193,19.141867,Normal
8177,68.85894,135.3170,1.749017,61.378709,20.064560,Normal
5844,71.68363,129.1334,1.820764,58.573877,17.668373,Mild Thinness
5534,64.70508,111.3172,1.643509,50.492591,18.693192,Normal
1917,70.15540,117.3159,1.781947,53.213554,16.758403,Moderate Thinness
2304,69.21780,142.8796,1.758132,64.809044,20.966821,Normal
20040,70.76552,142.9296,1.797444,64.831723,20.066736,Normal
5228,70.30094,148.0703,1.785644,67.163504,21.064136,Normal


Based on that, one could do lots of things like, `value_counts`:

In [21]:
df_socr['classification'].value_counts()

classification
Normal               17551
Mild Thinness         5780
Moderate Thinness     1262
Severe Thinness        401
Overweight               6
Name: count, dtype: int64

### Dropping columns

We might choose to drop the previous columns as well:

In [22]:
df_socr = df_socr.drop(
    columns=['Height(Inches)', 'Weight(Pounds)'],
)
df_socr.head()

,Height(meters),Weight(kilograms),BMI,classification
Index,,,,
1,1.670896,51.252494,18.357609,Mild Thinness
2,1.816486,61.909547,18.762615,Normal
3,1.762728,69.411778,22.338940,Normal
4,1.732702,64.562199,21.504569,Normal
5,1.721810,65.452010,22.077625,Normal


### Rename columns

Let's say we want to rename from `meters` to `m`, and from `kilograms` to `kg` instead:

In [23]:
df_socr = df_socr.rename(
    columns={
        'Height(meters)': 'Height(m)',
        'Weight(kilograms)': 'Weight(kg)',
    }
)
df_socr.head()

,Height(m),Weight(kg),BMI,classification
Index,,,,
1,1.670896,51.252494,18.357609,Mild Thinness
2,1.816486,61.909547,18.762615,Normal
3,1.762728,69.411778,22.338940,Normal
4,1.732702,64.562199,21.504569,Normal
5,1.721810,65.452010,22.077625,Normal


## Save to disk

In [25]:
df_socr.to_csv('SOCR-HeightWeight_modified.csv',)

## Key Takeaways

- **Vectorized Operations:** Arithmetic and logical operations between columns or between columns and scalars are applied elementwise (“broadcasting”).

- **Modifying DataFrames:**
  - Add new columns by assignment (e.g. `df["BMI"] = ...`).
  - Apply functions on values of a column using `apply(func)`
  - Drop columns with `df.drop(columns=[...])`.
  - Rename columns with `df.rename(columns={...})`.


These core tools and concepts are essential for effective data exploration, cleaning, and analysis using pandas.